# Blend tuning: TAPT XLM-R-large + SVM

The TAPT-adapted XLM-R-large (CV 0.9272 — best single model, beat plain XLM-R-large on all 5 folds by +0.005-0.008) paired with the linear SVM, its only permitted partner at 559,893,507 params (~562M total with the SVM, under the 600M cap).

Also evaluates temperature scaling for this pairing (rejected for the plain-large blend, but the TAPT model's calibration may differ), using the nested fold-wise protocol.

References: TAPT-large solo 0.9272; plain large+SVM blend 0.9238 (w=0.4); 3-way blend + T 0.9254.

## Load and align OOF predictions

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

LABELS = ["technical_support", "customer_support", "other"]
labels_arr = np.array(LABELS)
EPS = 1e-12

train = pd.read_csv("data/train.csv")
linear_oof = pd.read_csv("artifacts/oof_predictions/linear_oof_predictions.csv")
tapt_oof = pd.read_csv("artifacts/oof_predictions/xlmrlarge_tapt_oof_predictions.csv")
for df_ in (linear_oof, tapt_oof):
    assert (df_["id"] == train["id"]).all()

svm_scores = linear_oof[[f"score_{l}" for l in LABELS]].to_numpy()
svm_probs = np.exp(svm_scores) / np.exp(svm_scores).sum(axis=1, keepdims=True)
tapt_probs = tapt_oof[[f"prob_{l}" for l in LABELS]].to_numpy()
y_true = train["label"].to_numpy()

fold_of_row = np.zeros(len(train), dtype=int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (_, val_idx) in enumerate(skf.split(train, train["label"]), start=1):
    fold_of_row[val_idx] = fold_idx

def macro_f1_probs(probs, mask=None):
    if mask is None:
        mask = np.ones(len(y_true), dtype=bool)
    return f1_score(y_true[mask], labels_arr[probs[mask].argmax(axis=1)], average="macro")

print(f"SVM alone:        {macro_f1_probs(svm_probs):.4f}")
print(f"TAPT-large alone: {macro_f1_probs(tapt_probs):.4f}")

SVM alone:        0.8879
TAPT-large alone: 0.9272


## Disagreement: TAPT-large vs SVM

In [2]:
tapt_pred = labels_arr[tapt_probs.argmax(axis=1)]
svm_pred = labels_arr[svm_probs.argmax(axis=1)]
disagree = tapt_pred != svm_pred

print(f"Disagreement rate: {disagree.mean():.1%} ({disagree.sum()} rows)")
print(f"When they disagree - TAPT-large correct: {(tapt_pred[disagree] == y_true[disagree]).mean():.1%}")
print(f"When they disagree - SVM correct:        {(svm_pred[disagree] == y_true[disagree]).mean():.1%}")
print(f"(reference: vs plain large the SVM was right 32.0% of disagreements)")

Disagreement rate: 8.4% (1490 rows)
When they disagree - TAPT-large correct: 66.4%
When they disagree - SVM correct:        27.9%
(reference: vs plain large the SVM was right 32.0% of disagreements)


## Weight sweep: blend = w * tapt + (1 - w) * svm

In [3]:
results = []
for w in np.arange(0.0, 1.001, 0.05):
    blend = w * tapt_probs + (1 - w) * svm_probs
    results.append({"w_tapt": round(w, 2), "macro_f1": macro_f1_probs(blend)})

blend_df = pd.DataFrame(results)
best = blend_df.loc[blend_df["macro_f1"].idxmax()]
print(blend_df.to_string(index=False))
print(f"\nBest: w_tapt={best['w_tapt']}, Macro F1={best['macro_f1']:.4f}")
print("References: TAPT solo 0.9272 | plain large+SVM 0.9238 | 3-way+T 0.9254")

 w_tapt  macro_f1
   0.00  0.887893
   0.05  0.897609
   0.10  0.907371
   0.15  0.915358
   0.20  0.920991
   0.25  0.924955
   0.30  0.927188
   0.35  0.928158
   0.40  0.928382
   0.45  0.928213
   0.50  0.927920
   0.55  0.927648
   0.60  0.927321
   0.65  0.927471
   0.70  0.927208
   0.75  0.927280
   0.80  0.927065
   0.85  0.927209
   0.90  0.927233
   0.95  0.927354
   1.00  0.927204

Best: w_tapt=0.4, Macro F1=0.9284
References: TAPT solo 0.9272 | plain large+SVM 0.9238 | 3-way+T 0.9254


## Nested check: does temperature scaling help this pairing?

Rejected for plain large+SVM (single-transformer blends: the weight absorbs the calibration correction), but cheap to re-verify on the TAPT model under the honest nested protocol.

In [4]:
def temp_scale(p, T):
    q = np.power(p + EPS, 1.0 / T)
    return q / q.sum(axis=1, keepdims=True)

TEMPERATURES = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0]
W_GRID = np.round(np.arange(0.0, 1.001, 0.05), 2)

def nested(with_temp):
    fold_scores = []
    chosen = []
    for fold_idx in range(1, 6):
        val_mask = fold_of_row == fold_idx
        fit_mask = ~val_mask
        best_cfg, best_fit = None, -1
        temps = TEMPERATURES if with_temp else [1.0]
        for T in temps:
            scaled = temp_scale(tapt_probs, T)
            for w in W_GRID:
                s = macro_f1_probs(w * scaled + (1 - w) * svm_probs, fit_mask)
                if s > best_fit:
                    best_fit, best_cfg = s, (T, w)
        T, w = best_cfg
        fold_scores.append(macro_f1_probs(w * temp_scale(tapt_probs, T) + (1 - w) * svm_probs, val_mask))
        chosen.append(best_cfg)
    return fold_scores, chosen

plain_scores, plain_cfg = nested(with_temp=False)
temp_scores, temp_cfg = nested(with_temp=True)
for i in range(5):
    print(f"fold {i+1}: plain {plain_scores[i]:.4f} (w={plain_cfg[i][1]})  |  +T {temp_scores[i]:.4f} (T={temp_cfg[i][0]}, w={temp_cfg[i][1]})")
print(f"mean:   plain {np.mean(plain_scores):.4f}  |  +T {np.mean(temp_scores):.4f}")

fold 1: plain 0.9347 (w=0.4)  |  +T 0.9365 (T=20.0, w=0.85)
fold 2: plain 0.9259 (w=0.4)  |  +T 0.9264 (T=20.0, w=0.8)
fold 3: plain 0.9326 (w=0.4)  |  +T 0.9340 (T=20.0, w=0.8)
fold 4: plain 0.9310 (w=0.4)  |  +T 0.9320 (T=20.0, w=0.8)
fold 5: plain 0.9177 (w=0.4)  |  +T 0.9178 (T=12.0, w=0.75)
mean:   plain 0.9284  |  +T 0.9293


## Classification report at the adopted configuration

In [5]:
# W_TAPT set from the weight-sweep results above (w_tapt=0.4, the sweep's peak)
W_TAPT = float(best["w_tapt"])
final_blend = W_TAPT * tapt_probs + (1 - W_TAPT) * svm_probs
final_preds = labels_arr[final_blend.argmax(axis=1)]
print(f"Configuration: w_tapt={W_TAPT} (no temperature unless the nested check clearly said otherwise)")
print(classification_report(y_true, final_preds, digits=4))

Configuration: w_tapt=0.4 (no temperature unless the nested check clearly said otherwise)
                   precision    recall  f1-score   support

 customer_support     0.8836    0.9059    0.8946      3327
            other     0.9451    0.9111    0.9278      3800
technical_support     0.9604    0.9651    0.9627     10642

         accuracy                         0.9425     17769
        macro avg     0.9297    0.9274    0.9284     17769
     weighted avg     0.9427    0.9425    0.9425     17769

